### Purpose

거래처 관리용 DynamoDB 테이블 `dscustomers` + Lambda `dscustomerDynamoDB` 를 만들고
API Gateway `jyipsj28s9` 에 `/dscustomer` (GET/POST/PUT/DELETE) 로 연결한다.

- 소스: `dscustomerDynamoDB/lambda_function.py`
- 구성은 기존 `/dschemical` 과 동일 (인증 없음, 패스스루 매핑 템플릿, CORS `*`, 역할 `DSmapLambda`)
- 여러 번 실행해도 된다 (있으면 갱신)
- **마지막 셀(`dev` 스테이지 배포)은 API 전체를 다시 배포한다** — 콘솔에서 배포 안 한 변경이 있으면 같이 반영됨
- 데이터 적재/갱신은 웹(거래처정보 > 이카운트 엑셀 비교)에서 한다

In [ ]:
# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 - boto3 보다 먼저 적용해야 함
import truststore
truststore.inject_into_ssl()

import json
import boto3

boto3.setup_default_session(profile_name='default', region_name='us-east-1')
from deploy_helpers import (deploy_lambda, ensure_resource, get_request_template, setup_lambda_method,
                            setup_options, grant_invoke, deploy_stage_and_check)

Settings

In [ ]:
TABLE_NAME = 'dscustomers'
TABLE_KEY = 'custcd'

FUNCTION_NAME = 'dscustomerDynamoDB'
SOURCE_FILE = './dscustomerDynamoDB/lambda_function.py'
ROLE_ARN = 'arn:aws:iam::199525800117:role/DSmapLambda'   # dynamodb:* on table/* 권한 있음

API_ID = 'jyipsj28s9'
STAGE = 'dev'
RESOURCE_PATH = 'dscustomer'
METHODS = ['GET', 'POST', 'PUT', 'DELETE']
TEMPLATE_FROM = '/dschemical'

ACCOUNT_ID = ROLE_ARN.split(':')[4]

1. DynamoDB 테이블 (on-demand, dschemicals 와 같은 방식)

In [ ]:
ddb = boto3.client('dynamodb')
if TABLE_NAME in ddb.list_tables()['TableNames']:
    print('테이블 있음:', TABLE_NAME)
else:
    ddb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{'AttributeName': TABLE_KEY, 'KeyType': 'HASH'}],
        AttributeDefinitions=[{'AttributeName': TABLE_KEY, 'AttributeType': 'S'}],
        BillingMode='PAY_PER_REQUEST',
    )
    ddb.get_waiter('table_exists').wait(TableName=TABLE_NAME)
    print('테이블 생성:', TABLE_NAME)
ddb.describe_table(TableName=TABLE_NAME)['Table']['ItemCount']

2. Lambda

In [ ]:
function_arn = deploy_lambda(FUNCTION_NAME, SOURCE_FILE, ROLE_ARN,
                             description='거래처 CRUD (DynamoDB dscustomers)', timeout=29, memory=256)
function_arn

3. API Gateway `/dscustomer` + 권한

In [ ]:
resource_id = ensure_resource(API_ID, RESOURCE_PATH)
request_template = get_request_template(API_ID, TEMPLATE_FROM)

for m in METHODS:
    setup_lambda_method(API_ID, resource_id, m, function_arn, request_template)
setup_options(API_ID, resource_id, METHODS)
grant_invoke(FUNCTION_NAME, API_ID, ACCOUNT_ID, RESOURCE_PATH)

4. `dev` 스테이지 배포 + 확인

In [ ]:
res_ = deploy_stage_and_check(API_ID, STAGE, RESOURCE_PATH, f'add /{RESOURCE_PATH}')
body = json.loads(res_['body'])
print('statusCode', res_['statusCode'], '/', f'{len(body)}건' if isinstance(body, list) else body)